Si mediste en el terreno, o tenes una altura de referencia que se encuentra en tu Raster:

# CORRIGIR DEMS A ALTURA CONOCIDA
(posiblemente tengas que modificar ambos DEMs!!!)

OJO, buscar las indicaciones ***# MODIFICAR*** porque son las variables y datos que **tiene que cambiar el usuario del código**

In [ ]:
from termcolor import colored

import numpy as np
import pandas as pd
import geopandas as gpd
from osgeo import gdal
import os
from shapely.geometry import Point

from matplotlib import pyplot as plt
import matplotlib.image as mpimg

import rasterio
from rasterio.mask import mask
from rasterio.plot import show
from rasterio.plot import show_hist

#import funciones as fn

import sys
sys.path.append('../')

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Defino la funcion que necesito:
def Histograma(banda_del_arreglo, banda='banda', p=0, nodata=None, figsize=(12,6)):
    if np.ma.isMaskedArray(banda_del_arreglo):
        band = banda_del_arreglo.compressed()
    elif nodata is not None:
        band = banda_del_arreglo[banda_del_arreglo != nodata]
    else:
        band = banda_del_arreglo.ravel()  # usamos todos los datos tal cual

    p_b_min = np.percentile(band, p)
    p_b_max = np.percentile(band, 100-p)
    
    plt.figure(figsize=figsize)
    plt.hist(band, bins=100)
    plt.axvline(p_b_min, color='red', linestyle='--', label=f'Percentil {p}%')
    plt.axvline(p_b_max, color='black', linestyle='--', label=f'Percentil {100-p}%')
    plt.legend()
    plt.title(f'Histograma {banda}')
    plt.show()

In [ ]:
# MODIFICAR direcciorios, segun cual elijas mover (o los 2) (*)

dir_DEM_o = 'C:/' # MODIFICAR

dir_DEM_proc = 'C:/DEM_procesando' # MODIFICAR

DEM_i = 'tu_DEM_fechaInicial' # MODIFICAR
DEM_f = 'tu_DEM_fechaFinal' # MODIFICAR

archivo_H_ref = f'{dir_DEM_proc}/shapes/puntos_altura.shp' # MODIFICAR
#archivo_H_ref_pol = f'{dir_DEM_proc}/shapes/circulo_altura.shp' # No es necesario

archivo_DEM_i = f'{dir_DEM_o}/{DEM_i}.tif'
archivo_DEM_f = f'{dir_DEM_o}/{DEM_f}.tif'
archivo_DEM_i_2 = f'{dir_DEM_proc}/{DEM_i}_movido.tif' # ó puede ser  = archivo_DEM_i # Si no estaban desplazados entre ellos en la horizontal
archivo_DEM_i_2b = f'{dir_DEM_proc}/{DEM_i}_movido_nan.tif' # ó puede ser  = archivo_DEM_i # Si no estaban desplazados entre ellos en la horizontal
archivo_DEM_f_2 = f'{dir_DEM_proc}/{DEM_f}_movido.tif' # ó puede ser  = archivo_DEM_f # Si no estaban desplazados entre ellos en la horizontal
archivo_DEM_f_2b = f'{dir_DEM_proc}/{DEM_f}_movido_nan.tif' # ó puede ser  = archivo_DEM_f # Si no estaban desplazados entre ellos en la horizontal

archivo_DEM_i_3 = f'{dir_DEM_proc}/{DEM_i}_igualadoH.tif' # ó puede ser  = archivo_DEM_i # ó puede ser  = archivo_DEM_i_2 # ó puede ser  = archivo_DEM_i_2b
archivo_DEM_f_3 = f'{dir_DEM_proc}/{DEM_f}_igualadoH.tif' # ó puede ser  = archivo_DEM_f # ó puede ser  = archivo_DEM_f_2 # ó puede ser  = archivo_DEM_f_2b

archivo_DEM_i_4 = f'{dir_DEM_proc}/{DEM_i}_corregidoH.tif'
archivo_DEM_f_4 = f'{dir_DEM_proc}/{DEM_f}_corregidoH.tif'

Te recomiendo que vayas indentando con # los archivos que no corregis/moves

**Selecciona el DEM a mover verticalmente a tu referencia,**  
lo esperado aca es que sea el corregido en la horizontal (ver código 01)  
(Si no igualaste las alturas de los DEM _i y _f en el código 02, podemos mover ambos a tu referencia, corriendo 2 veces este .py)

In [ ]:
archivo_DEM = archivo_DEM_i  # MODIFICAR por el que te sea de interés para hacer el balance de masa en el siguiente código!!!
archivo_DEM_salida = archivo_DEM_i_4  ó # MODIFICAR cual moveras primero y luego segundo en este código!!!

# Tomar las alturas de tu referencia y del DEM

### 3 Opciones para que elijas cual aplicar:
(si aplicas la 2da, tenes que aplicar la 3ra luego!, *no te olvides de des-indentar la exportación del shapefile*)

* 1) Si no tenes la referencia en shapefile y sabes el valor del punto y determinaste la H del DEM en ese punto en un visor GIS, puedes directamente:

In [ ]:
# MODIFICAR:
altura_DEM = 0 # por tu_valor
# MODIFICAR:
altura_puntoGPS = 0 # por tu_valor

* 2) Si no tenes la referencia en shapefile y solo tenes un valor de altura y su posición en Lat y Long, te dejo unas lineas de código para que crees tu shape:

In [ ]:
# Datos del punto de referencia
Lat = -33 # MODIFICAR
Long = -58 # MODIFICAR
H = 5
geometry = [Point(Long, Lat)]
gdf = gpd.GeoDataFrame({
    'ID': [1],
    'ELEV': [H]
}, geometry=geometry, crs="EPSG:4326")  # WGS84
# Exportar a shapefile
#gdf.to_file(archivo_H_ref)  # MODIFICAR des-indenta si es que no tenes tu shape!!!
# Seguis a la 3)

* 3) Si tenes el punto de referencia con altura en un shapefile:

In [ ]:
# Abrir el shapefile del punto de referencia
gdf_puntoGPS = gpd.read_file(archivo_H_ref)
# (Asegurarse de que está en un sistema de coordenadas planas)
if gdf_puntoGPS.crs.is_geographic:
    gdf_puntoGPS = gdf_puntoGPS.to_crs(gdf_puntoGPS.estimate_utm_crs()) # Proyectar a UTM automáticamente (usa la zona más cercana al punto)

# Tomar la ultura del pto GPS del shape
gdf_puntoGPS_1 = gdf_puntoGPS[gdf_puntoGPS['ID'] == 1] # MODIFICAR x el atributo de tu shape
altura_puntoGPS = gdf_puntoGPS_1.iloc[0]['ELEV'] # MODIFICAR x el atributo de tu shape
print(f"📏 Altura del punto GPS: {altura_DEM_f:.2f}")

# Crear el buffer de 5 metros al rededor del pto GPS
gdf_buffer = gdf_puntoGPS.copy()
gdf_buffer["geometry"] = gdf_buffer.geometry.buffer(5)
# Podríamos exportar el buffer como shapefile:
#gdf_buffer.to_file(archivo_H_ref_pol) # No es necesario exportar el buffer, pero si así lo queres, desindenta

In [ ]:
with rasterio.open(archivo_DEM) as src:
    # (Asegurar que los CRS coinciden:)
    if gdf_buffer.crs != src.crs:
        gdf_buffer = gdf_buffer.to_crs(src.crs)
    # Recortar el DEM por el buffer
    out_image, _ = mask(src, gdf_buffer.geometry, crop=True)
    out_image = out_image.astype(float)  # por si hay enteros y luego hay NaN
    nodata = src.nodata # Toma el valor nodata
    
# Calcular la altura media del DEM cortado por el buffer al rededor del punto GPS, ignorando nodata
if nodata is not None:
    altura_DEM = np.mean(out_image[out_image != nodata])
else:
    altura_DEM = np.mean(out_image)
print(f"📏 Valor medio de los píxeles del DEM_f dentro del buffer al rededor del punto GPS: {altura_DEM:.2f}")

# Determinar la diferencia entre referencia y DEM

In [ ]:
diferencia = altura_DEM - altura_puntoGPS

# Bajo/Subo DEM

In [ ]:
DEM = gdal.Open(archivo_DEM) # arriba se define cual
gt_DEM = DEM.GetGeoTransform()
src_DEM = DEM.GetProjection()
DEM_arr = DEM.ReadAsArray()

# Valores
valor_maximo = np.nanmax(DEM_arr)
valor_minimo = np.nanmin(DEM_arr)
print("Valor máximo del pixel:", valor_maximo)
print("Valor mínimo del pixel:", valor_minimo)

In [ ]:
# Enmascarar los valores sin datos iguales a -9999 considerandolos no válidos
DEM_masked = np.ma.masked_equal(DEM_2017_arr, -9999)
# Calcula el valor máximo y mínimo, ignorando los valores enmascarados (-9999)
valor_maximo = np.max(DEM_masked)
valor_minimo = np.min(DEM_masked)
print("Valor máximo del pixel (excluyendo -9999):", valor_maximo)
print("Valor mínimo del pixel (excluyendo -9999):", valor_minimo)

In [ ]:
prom_DEM_masked = DEM_masked.mean()

plt.figure(figsize = (12,6))
plt.imshow(DEM_masked, vmin = 0, vmax = valor_maximo, cmap = 'viridis')
plt.title("DEM_masked")
plt.show()

print('Promedios de alturas para DEM_masked = ', prom_DEM_2017_masked)

In [ ]:
# Corro la función de Histograma
histo_DEM_masked = Histograma(DEM_masked, banda='DEM_masked', p=5, figsize=(12,6))

In [ ]:
################################################################################
########################   Ahora si, Bajo/Subo el DEM   ########################
################################################################################

DEM_masked_corregido = DEM_masked - diferencia 

In [ ]:
prom_DEM_masked_corregido = DEM_masked_corregido.mean()
print('Promedios de alturas para DEM_masked_corregido = ', prom_DEM_masked_corregido)
histo_DEM_masked_corregido = Histograma(DEM_masked_corregido, banda='DEM (Filtrado y Corregido en H)', p=5, figsize=(12,6))

In [ ]:
# Exportar el DEM subido/bajado:
filas = DEM_masked_corregido.shape[0]
columnas = DEM_masked_corregido.shape[1]
bandas = 1
driver = gdal.GetDriverByName('GTiff')
DEM_masked_corregido = driver.Create(archivo_DEM_salida, columnas, filas, bandas, gdal.GDT_Float32)
DEM_masked_corregido.SetProjection(src_DEM_2017)
DEM_masked_corregido.SetGeoTransform(gt_DEM_2017)
#print(DEM_masked_corregido.GetGeoTransform())
# Establece NaN como el valor de los datos no válidos para la banda:
banda = DEM_masked_corregido.GetRasterBand(1)
banda.SetNoDataValue(float('nan'))  # Aquí se especifica NaN como valor de los datos no válidos
# Escribir los datos al archivo:
banda.WriteArray(DEM_masked_corregido[:,:])
# Limpieza
del banda  # Importante liberar la banda antes del dataset si realizas operaciones adicionales
del DEM_masked_corregido  # Cierra y guarda el archivo

# Bajar/Subir el otro DEM

Solo debes volver a correr el código completo, modificando el ***'archivo_DEM'*** y el ***'archivo_DEM_salida'*** (ver ***# MODIFICAR*** a lo largo del codigo, puedes usar *ctrl-f*). 

*Te recomiendo hagas una copia del *.py para cada DEM.*